# Column reference

**Current state** — `direction` is the open Supertrend flip and sets the structure:
**UP → bull put**, **DOWN → bear call**.

| Column | Meaning |
|---|---|
| `symbol` · `direction` | Ticker, and which way the open flip points |
| `trend_age_days` | Calendar days since the flip (weekends count) |
| `spot` · `flip_level` | Latest close, and the Supertrend level that would invalidate the trend |
| `buffer_to_flip_pct` | Distance from spot to that level. **Small = about to invalidate** (NIFTY sat at 0.27% while topping the containment table) |
| `moved_since_entry_pct` | Signed *for* the position: positive = trend working, negative = underwater |
| `sector` · `beta` | From a hand-maintained map; beta vs NIFTY over ~250 days. Beta makes the 50/50 split *shock*-balanced, not just count-balanced |
| `realized_vol_pct` · `realized_vol_rank` | Annualized `window`-day realized vol, and where TODAY's value ranks (0-100) against this stock's own trailing ~2yr history. **Backfillable interim proxy for IV rank — not IV itself, no established threshold, informational only for now** (see Stage 8f) |

**Duration context** — descriptive only.

| Column | Meaning |
|---|---|
| `median_duration` | This stock's typical flip length |
| `past_median` · `past_75th_pct` · `past_90th_pct` | Has the open trend outlived that checkpoint? |
| `runway_left_vs_median` · `_p75` · `_p90` | `checkpoint − age` in signed days; negative = past it |

> `reversion_test.py` and `exhaustion_test.py` both tested whether crossing these
> predicts a reversal. **It does not.** `past_median = True` is not a warning and a
> negative runway is not an exit signal.

**Does the signal work here?** — all on closed flips in the *currently open* direction.

| Column | Meaning |
|---|---|
| `dir_flips` | Sample size; under 12 the edge columns return `None` |
| `dir_win_rate` | % of those flips that moved as the signal said |
| `dir_median_move` | Median favourable move. **Standalone for this side — not a comparison to the opposite side.** Negative = the typical flip goes the wrong way |

**Strike selection** — containment is the primary metric. A credit spread is a
"price won't breach my strike" bet, not a directional one.

| Column | Meaning |
|---|---|
| `contained_5pct` · `contained_6pct` | % of same-direction flips whose worst adverse excursion stayed inside that buffer |
| `worst_adverse` | Largest adverse excursion in 4yr — watch for fat tails behind good containment |
| `worst_gap` | Largest overnight gap against the position (open vs prev close) |
| `worst_day` | Largest full close-to-close day against it. Exists because `worst_gap` misses intraday collapses — PIIND's 8.97% fall was only a 1.28% gap |
| `pct_within_30d` | % resolved inside 30 days — matters for monthly expiries |
| `short_strike` · `strike_buffer_pct` | Strike implied by this stock's own containment, off live spot. Says nothing about whether the premium there is worth taking (needs IV — not built) |

**Was this flip normal?**

| Column | Meaning |
|---|---|
| `entry_day_move` | Signed move on the day the open flip was born |
| `born_on_outlier_day` | `abs(entry_day_move) > 4%`. The containment stats above come from *ordinary* flips and do not describe a trend that began mid-crash |

**Wrong half of a good stock?**

| Column | Meaning |
|---|---|
| `opp_win_rate` · `opp_median_move` | The direction you are **not** trading. **Lower is not better** — these say nothing about the current trade. Their only job: current weak + opposite strong → watchlist; weak both ways → no edge |

**Verdicts**

| Column | Meaning |
|---|---|
| `edge_flag` | Empty = tradeable · `⚠ weak side` = watchlist · `⚠ NO EDGE EITHER SIDE` = fails both ways · `⚠ flip born on an outlier day` appended to any of these · `insufficient history` |
| `setup_score` | 0–100 composite (containment ×3, gap ×2, day ×2, win ×2, move ×1). **Relative to today's screen only** — not comparable across days |
| `reason` | Plain-language why, built from this stock's own numbers |

Bar for "has edge": `dir_win_rate >= 50` **and** `dir_median_move > 0`. Deliberately
plain, not optimised — nothing here is walk-forward validated. Names on the boundary
(BOSCHLTD `50.0/+0.08` passes, KOTAKBANK `50.0/−0.10` fails) are indistinguishable.

---

**Caveats**

1. **In-sample and descriptive** — same ~4 years the ranking is built from, no
   walk-forward validation. What the signal *has* done, not what it will do.
2. **Duration columns are pooled; edge columns are split by direction.**
3. **Containment spans the whole flip, not 30 days** — conservative.
4. **Not here at all:** earnings dates, real IV rank (a forward IV-history recorder started — `scripts/record_iv_snapshot.py` — but needs months to accumulate a usable percentile; `realized_vol_rank` above is the interim proxy), options liquidity, exits.


In [1]:
# It reads the saved credit-spread ranking, drops anything flagged for gap risk, 
# takes the top 50 by score, 
# then force-adds NIFTY/BANKNIFTY (same gap-risk filter applied to them too), 
# and merges into one deduplicated final_candidates list. No API calls here — pure filtering on the 
# last saved ranking file.


# autoreload: re-imports any changed src/*.py automatically, so an old kernel
# can never run stale module code. Without this, editing src/ and re-running
# without a kernel restart silently uses the OLD version — which is how 31
# reports once got rewritten missing their new columns.
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
from src.daily_screener import screen_list

credit_spread_ranking = pd.read_excel("../exports/RANKING_credit_spreads.xlsx")

# Keep only gap-risk-clean stocks, take the top 50
clean_candidates = credit_spread_ranking[
    credit_spread_ranking["gap_risk_flag"].isna() | (credit_spread_ranking["gap_risk_flag"] == "")
]
top_50 = clean_candidates.head(50)

# Always include NIFTY and BANKNIFTY, gap-risk-checked the same way
must_include = credit_spread_ranking[credit_spread_ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[must_include["gap_risk_flag"].isna() | (must_include["gap_risk_flag"] == "")]

final_candidates = pd.concat([top_50, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)

print("SORTED BY: credit_spread_score, descending (inherited from the ranking file).")
print("           NIFTY/BANKNIFTY appended at the end if not already in the top 50.")
final_candidates

SORTED BY: credit_spread_score, descending (inherited from the ranking file).
           NIFTY/BANKNIFTY appended at the end if not already in the top 50.


,symbol,credit_spread_score,gap_risk_flag,day_risk_flag,Total flips (closed),Median duration (days),% completed within a monthly cycle,Median max_adverse_pct,Worst max_adverse_pct (4yr),Worst single gap (4yr),Worst single-day move against (4yr),current_trend_age_days,trend_age_note
0,NESTLEIND,92.5,NaN,NaN,55,24.2,58.2,2.73,5.96,4.37,3.61,14,NaN
1,ITC,86.9,NaN,NaN,53,23.2,62.3,2.48,9.56,4.68,4.89,20,NaN
2,SUNPHARMA,84.4,NaN,NaN,53,20.0,66.0,2.57,7.31,3.53,4.51,29,NaN
3,DRREDDY,84.0,NaN,NaN,58,22.2,67.2,2.84,7.88,4.27,4.88,21,NaN
4,ULTRACEMCO,83.7,NaN,NaN,67,21.0,65.7,2.41,7.21,3.61,6.86,21,NaN
5,COLPAL,81.8,NaN,NaN,59,19.2,66.1,3.12,5.84,2.32,4.78,28,NaN
6,HAVELLS,80.2,NaN,NaN,63,22.0,68.3,3.28,6.58,4.42,4.54,8,NaN
7,TECHM,78.2,NaN,NaN,66,19.6,62.1,3.16,6.85,4.37,5.81,0,NaN
8,HCLTECH,78.2,NaN,NaN,64,18.2,65.6,2.80,6.33,4.26,5.36,0,NaN
9,BANKNIFTY,77.7,NaN,NaN,70,19.6,71.4,1.52,7.15,3.73,3.96,29,NaN


In [2]:
# For each of the ~50 symbols it looks up the right instrument 
# token (handling NIFTY/BANKNIFTY specially since indices live in a different token space), 
# fetches only the recent catch-up window of 4H candles from Kite (not the full 2022-to-now
# history - process_stock_incremental() merges this onto existing saved data instead of
# replacing it, same fix already applied to the debit spread/option buying/pledged pipelines,
# here for efficiency rather than a data-loss bug),
# reruns Supertrend → flip detection → monthly range → summary stats on the FULL merged
# history, and updates that symbol's _full_report.xlsx. This is the only block that talks to the API

from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.scale_pipeline import process_stock_incremental
from datetime import datetime

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        process_stock_incremental(symbol, token, kite, to_date=datetime.now())
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/NESTLEIND_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/NESTLEIND_full_report.xlsx
[1/50] NESTLEIND — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ITC_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/ITC_full_report.xlsx
[2/50] ITC — refreshed
Fetching 2026-05-11 to 2026-09-08...


Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/SUNPHARMA_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/SUNPHARMA_full_report.xlsx
[3/50] SUNPHARMA — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/DRREDDY_4hour.csv


Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/DRREDDY_full_report.xlsx
[4/50] DRREDDY — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ULTRACEMCO_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/ULTRACEMCO_full_report.xlsx
[5/50] ULTRACEMCO — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/COLPAL_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/COLPAL_full_report.xlsx
[6/50] COLPAL — refreshed


Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/HAVELLS_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/HAVELLS_full_report.xlsx
[7/50] HAVELLS — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/TECHM_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/TECHM_full_report.xlsx
[8/50] TECHM — refreshed
Fetching 2026-05-11 to 2026-09-08...


Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/HCLTECH_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/HCLTECH_full_report.xlsx
[9/50] HCLTECH — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/BANKNIFTY_4hour.csv


...

Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ALKEM_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/ALKEM_full_report.xlsx
[38/50] ALKEM — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/LICHSGFIN_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/LICHSGFIN_full_report.xlsx
[39/50] LICHSGFIN — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/CONCOR_4hour.csv


Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/CONCOR_full_report.xlsx
[40/50] CONCOR — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/BAJAJ-AUTO_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/BAJAJ-AUTO_full_report.xlsx
[41/50] BAJAJ-AUTO — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/BAJAJHLDNG_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/BAJAJHLDNG_full_report.xlsx
[42/50] BAJAJHLDNG — refreshed
Fetching 2026-05-11 to 2026-09-08...


Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/POLYCAB_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/POLYCAB_full_report.xlsx
[43/50] POLYCAB — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/CHOLAFIN_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/CHOLAFIN_full_report.xlsx
[44/50] CHOLAFIN — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/LT_4hour.csv


Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/LT_full_report.xlsx
[45/50] LT — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/EICHERMOT_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/EICHERMOT_full_report.xlsx
[46/50] EICHERMOT — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ICICIGI_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/ICICIGI_full_report.xlsx
[47/50] ICICIGI — refreshed
[48/50] FINNIFTY — no token found, skipping
Fetching 2026-05-11 to 2026-09-08...


Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/FORTIS_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/FORTIS_full_report.xlsx
[49/50] FORTIS — refreshed
Fetching 2026-05-11 to 2026-09-08...
Saved 2311 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/RELIANCE_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/RELIANCE_full_report.xlsx
[50/50] RELIANCE — refreshed


In [3]:
# reads the just-refreshed reports and reports current state:
# for each symbol, current trend direction, how many days the open trend has run,
# whether that age has crossed the stock's own median/75th/90th percentile duration,
# and runway_left_vs_median/p75/p90 - the same comparison as a signed number of
# days instead of three separate booleans (negative = already past that checkpoint) -
# sorted so the youngest (most recently flipped) trends show first. Same caveat as
# before: crossing a checkpoint is NOT a reversal/exit signal (tested, see README) -
# read runway as "how this compares to history," not as a warning.
#
# NEW - directional edge. The credit spread ranking read in cell 1 is built purely
# from risk-containment metrics (median/worst adverse, worst gap, duration, % within
# a monthly cycle) POOLED across both directions, and contains no profitability term
# at all. So a stock whose UP flips work and whose DOWN flips don't scores exactly
# like one where both work, and the shortlist gives no way to tell them apart. These
# columns answer, for the side that is actually open right now:
#   dir_win_rate/dir_median_move  - did the underlying actually move the way the
#       signal said? (directional read)
#   contained_5pct/contained_6pct - did price stay inside that buffer? This is what a
#       credit spread actually needs, since the structure is a "won't breach my
#       strike" bet, not a directional one. Read containment as PRIMARY and win rate
#       as a secondary tilt.
#   opp_win_rate/opp_median_move  - the other side, for contrast
# Caveat: in-sample and descriptive, same family as the runway columns - computed on
# the same ~4 years the ranking is built from, no walk-forward validation. It says
# what the signal HAS done here, not what it WILL do.

from src.daily_screener import screen_list

status = screen_list(symbols_to_refresh)

print("SORTED BY: trend_age_days, ascending — most recently flipped first.")
status.sort_values("trend_age_days")


SORTED BY: trend_age_days, ascending — most recently flipped first.


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,runway_left_vs_p90,...,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move,edge_flag
36,TCS,DOWN,0,14.2,False,False,False,14.2,29.2,48.2,...,97.4,6.86,2.71,4.49,73.7,-0.60,False,53.8,0.26,
17,BOSCHLTD,UP,0,20.2,False,False,False,20.2,40.4,65.4,...,93.1,7.44,2.72,5.22,65.5,0.67,False,37.9,-1.12,
7,TECHM,DOWN,1,19.2,False,False,False,18.2,33.2,46.1,...,90.9,6.85,3.03,5.27,63.6,-1.93,False,67.6,1.87,
8,HCLTECH,DOWN,1,17.2,False,False,False,16.2,36.2,54.1,...,96.9,6.29,2.01,4.40,75.0,-0.94,False,60.6,1.53,⚠ weak side — opposite direction is the strong...
40,BAJAJ-AUTO,DOWN,1,15.2,False,False,False,14.2,30.0,44.3,...,100.0,5.98,2.62,5.05,73.5,-0.97,False,54.3,0.39,⚠ weak side — opposite direction is the strong...
18,PIIND,UP,4,22.2,False,False,False,18.2,35.9,75.0,...,92.3,7.60,4.07,10.18,69.2,2.70,False,69.2,2.75,
14,TORNTPHARM,DOWN,4,25.0,False,False,False,21.0,39.2,68.6,...,82.6,8.46,1.98,5.11,65.2,-1.55,False,54.2,1.16,⚠ weak side — opposite direction is the strong...
49,RELIANCE,UP,5,15.1,False,False,False,10.1,22.0,34.7,...,100.0,4.88,4.43,3.53,77.5,0.86,False,47.5,-0.22,⚠ NO EDGE EITHER SIDE — signal has not worked ...
34,JINDALSTEL,DOWN,5,21.0,False,False,False,16.0,31.6,49.8,...,77.4,10.29,3.88,7.70,74.2,-1.30,False,65.6,3.28,⚠ weak side — opposite direction is the strong...
3,DRREDDY,DOWN,5,22.2,False,False,False,17.2,33.4,50.2,...,93.1,6.40,4.27,4.88,72.4,-0.27,False,63.3,1.05,


In [4]:
# Splits the screen into the three buckets that call for DIFFERENT action, so a
# no-edge name can't sit next to a good one looking identical (which is exactly what
# the pooled ranking lets happen):
#
#   tradeable - the open direction has historically worked on this stock
#   watchlist - wrong side of a stock whose OTHER side works. Not a bad stock, just
#               the wrong half of it right now - revisit when it flips, don't discard
#   no_edge   - the signal has not worked in EITHER direction. This is the case the
#               ranking is blindest to: score only measures how well-behaved price
#               is, so a stock with tight, gap-free moves and a signal that's wrong
#               both ways scores well and looks like a good candidate. Waiting for a
#               flip does not fix these - the signal itself doesn't work here.
#
# Nothing is deleted - no_edge is returned, not dropped, so the exclusion stays
# visible and auditable. Same reasoning as the scorer's gap_risk_flag, which warns
# without removing the row.
#
# See the column reference at the top of this notebook for what each field means.

from src.daily_screener import split_by_edge

tradeable, watchlist, no_edge = split_by_edge(status)

edge_cols = ["symbol", "direction", "trend_age_days", "dir_flips", "dir_win_rate",
             "dir_median_move", "contained_5pct", "contained_6pct", "worst_adverse",
             "worst_gap", "worst_day", "pct_within_30d", "entry_day_move",
             "born_on_outlier_day", "opp_win_rate", "opp_median_move"]

print(f"=== NO EDGE EITHER SIDE ({len(no_edge)}) — do not trade, flipping won't help ===")
print("    SORTED BY: nothing — falls through in screen order (i.e. ranking order)")
display(no_edge[edge_cols])

print(f"\n=== WATCHLIST ({len(watchlist)}) — wrong side now, other side works ===")
print("    SORTED BY: opp_win_rate, descending — best flip candidates first")
display(watchlist[edge_cols].sort_values("opp_win_rate", ascending=False))

print(f"\n=== TRADEABLE ({len(tradeable)}) — open direction has worked historically ===")
print("    SORTED BY: contained_6pct desc, then dir_win_rate desc — containment is the")
print("               primary metric, so a top win rate can still sit low in this table")
display(tradeable[edge_cols].sort_values(["contained_6pct", "dir_win_rate"], ascending=False))

# Crash-born flips, called out separately. These sit INSIDE the buckets above (a
# stock with real edge still lands in tradeable) but the containment stats there
# are drawn from ordinary flips and do not describe a trend that began mid-crash -
# PIIND topped this shortlist on 2026-08-12 on a flip created that morning by a
# -8.97% fall. Printed last so it's the thing left on screen before you act.
outliers = status[status["born_on_outlier_day"] == True]
if len(outliers):
    print(f"\n=== ⚠ FLIPS BORN ON AN OUTLIER DAY ({len(outliers)}) — stats above do not describe these ===")
    print("    SORTED BY: entry_day_move, ascending — biggest fall first")
    display(outliers[["symbol", "direction", "trend_age_days", "entry_day_move",
                      "dir_win_rate", "contained_6pct", "worst_gap", "worst_day"]]
            .sort_values("entry_day_move"))


=== NO EDGE EITHER SIDE (6) — do not trade, flipping won't help ===
    SORTED BY: nothing — falls through in screen order (i.e. ranking order)


,symbol,direction,trend_age_days,dir_flips,dir_win_rate,dir_median_move,contained_5pct,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move
19,KOTAKBANK,UP,52,36,50.0,-0.10,91.7,94.4,6.78,2.52,5.26,77.8,3.72,False,38.9,-1.13
21,GRASIM,UP,34,33,45.5,-0.34,93.9,97.0,6.95,2.91,3.78,69.7,2.97,False,30.3,-1.31
30,HINDUNILVR,DOWN,42,34,44.1,-0.11,94.1,97.1,6.82,4.18,4.51,67.6,-7.07,True,42.9,-0.21
37,ALKEM,DOWN,32,30,46.7,-1.53,73.3,86.7,13.46,3.33,6.60,70.0,0.63,False,38.7,-0.46
46,ICICIGI,DOWN,13,31,48.4,-0.27,83.9,96.8,6.56,3.32,7.39,67.7,-1.12,False,37.5,-1.59
49,RELIANCE,UP,5,40,40.0,-0.84,100.0,100.0,4.88,4.43,3.53,77.5,0.86,False,47.5,-0.22



=== WATCHLIST (20) — wrong side now, other side works ===
    SORTED BY: opp_win_rate, descending — best flip candidates first


,symbol,direction,trend_age_days,dir_flips,dir_win_rate,dir_median_move,contained_5pct,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move
42,POLYCAB,DOWN,18,31,48.4,-1.37,71.0,80.6,12.72,4.66,6.01,80.6,-1.47,False,68.8,3.00
34,JINDALSTEL,DOWN,5,31,38.7,-0.84,58.1,77.4,10.29,3.88,7.70,74.2,-1.30,False,65.6,3.28
31,TVSMOTOR,DOWN,11,28,42.9,-0.93,85.7,89.3,8.40,4.45,5.06,78.6,-2.43,False,65.5,4.01
0,NESTLEIND,DOWN,21,27,44.4,-0.05,88.9,100.0,5.96,2.45,3.61,74.1,-0.63,False,64.3,1.88
13,BAJAJFINSV,DOWN,32,29,44.8,-0.77,82.8,89.7,7.86,3.53,4.91,58.6,-3.86,False,63.3,1.14
39,CONCOR,DOWN,11,33,42.4,-0.60,81.8,84.8,22.36,3.29,6.35,69.7,-1.60,False,61.8,1.25
8,HCLTECH,DOWN,1,32,46.9,-0.56,87.5,96.9,6.29,2.01,4.40,75.0,-0.94,False,60.6,1.53
5,COLPAL,DOWN,35,29,37.9,-1.12,89.7,100.0,5.51,2.12,3.59,65.5,-1.89,False,60.0,0.47
44,LT,DOWN,8,34,44.1,-0.60,91.2,97.1,11.66,3.02,5.18,85.3,-0.41,False,60.0,1.68
23,BHARTIARTL,DOWN,13,31,32.3,-1.34,87.1,93.5,12.98,4.25,4.31,80.6,-1.54,False,59.4,1.27



=== TRADEABLE (24) — open direction has worked historically ===
    SORTED BY: contained_6pct desc, then dir_win_rate desc — containment is the
               primary metric, so a top win rate can still sit low in this table


,symbol,direction,trend_age_days,dir_flips,dir_win_rate,dir_median_move,contained_5pct,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move
28,AXISBANK,UP,13,36,69.4,2.30,97.2,100.0,5.49,2.72,3.81,77.8,1.71,False,41.7,-1.31
9,BANKNIFTY,UP,35,35,68.6,1.31,97.1,100.0,5.54,2.37,2.25,68.6,1.81,False,51.4,0.78
22,JSWSTEEL,UP,15,32,62.5,0.84,90.6,100.0,5.90,3.31,4.39,56.2,1.98,False,40.6,-0.44
11,SHREECEM,DOWN,8,30,60.0,1.87,86.7,100.0,5.59,2.28,5.63,73.3,-0.79,False,54.8,1.20
35,TITAN,DOWN,5,32,59.4,0.79,96.9,100.0,5.65,2.94,5.75,81.2,-0.79,False,60.6,1.26
20,ICICIBANK,UP,12,35,57.1,0.77,97.1,100.0,5.22,2.95,3.13,71.4,0.89,False,42.9,-1.17
32,NIFTY,DOWN,21,38,55.3,0.18,97.4,100.0,5.95,2.32,3.16,81.6,-0.55,False,69.2,0.66
33,SENSEX,DOWN,21,38,55.3,0.48,97.4,100.0,5.55,2.31,3.24,81.6,-0.63,False,69.2,0.70
26,APOLLOHOSP,UP,24,33,54.5,0.31,78.8,100.0,5.95,4.34,5.19,63.6,3.95,False,39.4,-0.88
36,TCS,DOWN,0,38,60.5,1.56,94.7,97.4,6.86,2.71,4.49,73.7,-0.60,False,53.8,0.26



=== ⚠ FLIPS BORN ON AN OUTLIER DAY (5) — stats above do not describe these ===
    SORTED BY: entry_day_move, ascending — biggest fall first


,symbol,direction,trend_age_days,entry_day_move,dir_win_rate,contained_6pct,worst_gap,worst_day
30,HINDUNILVR,DOWN,42,-7.07,44.1,97.1,4.18,4.51
15,HDFCBANK,DOWN,50,-5.22,51.6,96.8,2.11,3.58
48,FORTIS,DOWN,27,-5.13,36.0,64.0,2.81,8.13
27,LUPIN,DOWN,48,-4.40,33.3,86.7,4.74,5.10
38,LICHSGFIN,UP,13,4.07,60.6,87.9,3.14,5.76


In [5]:
# ============================================================================
#  THE FINAL LIST — this is the only cell you need to read.
# ============================================================================
# Everything above is the working-out. This applies the whole filter chain and
# returns the book to trade, balanced 50/50 between bull puts and bear calls.
#
# Filters applied, in order (each one is reported below, nothing is silent):
#   1. no edge either side   - signal has never worked on this stock
#   2. wrong side            - open direction is this stock's weak half
#   3. crash-born flip       - flip created by an outlier day; the containment
#                              stats come from ordinary flips and don't apply
#   4. thin buffer           - spot too close to the level that invalidates it
#   5. trend underwater      - thesis already failing before entry
#   6. sector cap            - per DIRECTION (opposite bets on one sector hedge
#                              each other, they don't concentrate risk)
#   7. 50/50 direction       - so one market-wide shock can't hit everything
#
# Identical to `python scripts/daily_list.py`, which is the schedulable version.

import importlib
import src.daily_screener, src.market_context, src.entry_advisor
importlib.reload(src.daily_screener)
importlib.reload(src.market_context)
importlib.reload(src.entry_advisor)
from src.entry_advisor import build_entry_list

final, excluded, summary = build_entry_list(symbols_to_refresh, n_total=12, max_per_sector=2)

print(f"{summary['up']} BULL PUT  /  {summary['down']} BEAR CALL"
      f"      mean beta {summary['mean_beta_up']} UP vs {summary['mean_beta_down']} DOWN")
if summary["up"] != summary["down"]:
    print("  ** NOT BALANCED — too few DOWN-side candidates survived the filters.")
    print("     Trade fewer positions rather than forcing in a weak bear call.")
print("SORTED BY: direction (DOWN before UP, alphabetical — not a quality ordering),")
print("           then setup_score descending WITHIN each direction block.")
print("           Which names got PICKED was decided by setup_score desc overall.")
print()

display(final[["symbol", "direction", "trend_age_days", "sector", "short_strike", "spot",
               "buffer_to_flip_pct", "dir_win_rate", "dir_median_move",
               "contained_6pct", "worst_gap", "worst_day", "beta",
               "realized_vol_rank", "setup_score"]])

# The WHY for each, so the list can be acted on without re-deriving anything.
for _, r in final.iterrows():
    print(f"\n{r['symbol']} — short strike ~{r['short_strike']:,.1f} "
          f"({r['strike_buffer_pct']}% from spot)")
    print(f"   {r['reason']}")

print("\n" + "=" * 78)
print("EXCLUDED — considered and rejected")
for key, names in excluded.items():
    if names:
        print(f"  {key:24s} {', '.join(names)}")

print("\nSTILL MANUAL, not checked by any of the above:")
print("  earnings / corporate actions · IV rank · options liquidity · exits")
print("All statistics are in-sample and descriptive — what the signal HAS done,")
print("not what it will do.")


2 BULL PUT  /  6 BEAR CALL      mean beta 1.12 UP vs 0.91 DOWN
  ** NOT BALANCED — too few DOWN-side candidates survived the filters.
     Trade fewer positions rather than forcing in a weak bear call.
SORTED BY: direction (DOWN before UP, alphabetical — not a quality ordering),
           then setup_score descending WITHIN each direction block.
           Which names got PICKED was decided by setup_score desc overall.



,symbol,direction,trend_age_days,sector,short_strike,spot,buffer_to_flip_pct,dir_win_rate,dir_median_move,contained_6pct,worst_gap,worst_day,beta,realized_vol_rank,setup_score
33,SENSEX,DOWN,21,INDEX,81352.3,76747.47,1.59,55.3,0.48,100.0,2.31,3.24,1.01,0.4,70.6
11,SHREECEM,DOWN,8,CEMENT,24782.8,23380.00,4.07,60.0,1.87,100.0,2.28,5.63,1.01,33.6,69.4
32,NIFTY,DOWN,21,INDEX,25103.6,23682.60,1.54,55.3,0.18,100.0,2.32,3.16,1.00,1.0,68.1
36,TCS,DOWN,0,IT,2396.1,2260.50,4.05,60.5,1.56,97.4,2.71,4.49,0.81,65.0,65.0
1,ITC,DOWN,27,FMCG,283.8,265.25,2.23,61.5,0.40,92.3,2.05,4.10,0.66,30.6,60.6
12,DABUR,DOWN,55,FMCG,399.1,376.50,2.78,53.1,0.53,96.9,2.66,3.66,0.95,1.8,53.8
22,JSWSTEEL,UP,15,METALS,1235.0,1313.80,1.53,62.5,0.84,100.0,3.31,4.39,1.12,17.0,68.8
17,BOSCHLTD,UP,0,AUTO,45360.8,48775.00,5.04,51.7,0.30,93.1,2.72,5.22,1.12,44.2,30.9



SENSEX — short strike ~81,352.3 (6.0% from spot)
   BEAR CALL — trend is DOWN, sell calls above. Flip is 21d old vs a 16.0d median. DOWN flips here win 55.3% with a +0.48% median move (38 samples). 100.0% of them stayed inside 6% (worst 5.55%). Worst gap 2.31%, worst single day 3.24%. 81.6% resolved within 30d. Spot 76747.47 sits 1.59% from the 77968.59 flip level. Trend is working (+0.79% since flip). Beta 1.01. Realized vol 6.6% (annualized), compressed vs its own 2yr history (0.4th pct — proxy only, not IV, no established threshold yet).

SHREECEM — short strike ~24,782.8 (6.0% from spot)
   BEAR CALL — trend is DOWN, sell calls above. Flip is 8d old vs a 22.0d median. DOWN flips here win 60.0% with a +1.87% median move (30 samples). 100.0% of them stayed inside 6% (worst 5.59%). Worst gap 2.28%, worst single day 5.63%. 73.3% resolved within 30d. Spot 23380.0 sits 4.07% from the 24330.45 flip level. Trend is working (+1.83% since flip). Beta 1.01. Realized vol 18.6% (annualized), i